In [ ]:
# NC-TCN-20K KWS Fine-Tuning Comparison (Interspeech 2026)
# Real audio fine-tuning on Google Speech Commands V2.
#
# 3 algorithms compared:
# 1. Standard LoRA (rank=4) — baseline
# 2. NC-ALoRA-PR (Adaptive rank + Prototype Regularization) — novel
# 3. NC-OPAL (Prototype-Imprinted Init + Knowledge Distillation) — novel
#
# Task: Add new wake words (marvin, sheila, bed) to NC-TCN-20K
# while preserving base 12-class accuracy.

In [ ]:
# ============================================================
# 1. Setup: Clone repo & install dependencies
# ============================================================
import os
!git clone https://github.com/DrJinHoChoi/NC-KWS-FineTuning.git 2>/dev/null || true
%cd /content/NC-KWS-FineTuning
!pip install -q torch torchaudio numpy matplotlib pandas

# Verify repo structure
import sys
sys.path.insert(0, '/content/NC-KWS-FineTuning')
sys.path.insert(0, '.')
assert os.path.exists('nanomamba.py'), 'ERROR: nanomamba.py not found! Check git clone.'
assert os.path.exists('checkpoints/best.pt'), 'ERROR: checkpoint not found!'
print(f"CWD: {os.getcwd()}")
print(f"nanomamba.py: OK")
print(f"checkpoints/best.pt: OK")
!ls -la nanomamba.py checkpoints/ core/

In [ ]:
# ============================================================
# 2. Download Google Speech Commands V2 (~2.3GB)
# ============================================================
import os, tarfile, urllib.request

GSC_URL = 'http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz'
GSC_TAR = 'data/speech_commands_v0.02.tar.gz'
GSC_DIR = 'data/speech_commands_v0.02'

os.makedirs('data', exist_ok=True)
if not os.path.exists(GSC_DIR):
    if not os.path.exists(GSC_TAR):
        print('Downloading GSC V2 (~2.3GB)...')
        urllib.request.urlretrieve(GSC_URL, GSC_TAR)
        print('Download complete.')
    print('Extracting...')
    os.makedirs(GSC_DIR, exist_ok=True)
    with tarfile.open(GSC_TAR, 'r:gz') as tar:
        tar.extractall(GSC_DIR)
    print('Done!')
else:
    print(f'GSC V2 already at {GSC_DIR}')

kw_dirs = sorted([d for d in os.listdir(GSC_DIR)
                  if os.path.isdir(os.path.join(GSC_DIR, d)) and not d.startswith('_')])
print(f'{len(kw_dirs)} keywords: {kw_dirs}')

In [ ]:
# ============================================================
# 3. Write fine-tuning modules (self-contained for Colab)
# ============================================================
import os
os.makedirs('core', exist_ok=True)
for d in ['core']:
    init = os.path.join(d, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()

# --- Standard LoRA (kws_finetune.py) ---
KWS_FT_V1 = r'''
import os, sys, json, time, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F


PARENT = os.path.join(os.path.dirname(__file__), '..')

sys.path.insert(0, PARENT)

class LoRALinear(nn.Module):
    def __init__(self, original, rank=4, alpha=8):
        super().__init__()
        self.original = original
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(original.in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, original.out_features))
    def forward(self, x):
        return self.original(x) + (x @ self.lora_A @ self.lora_B) * self.scaling

class KWSFineTuner:
    SR = 16000
    BASE_LABELS = ['yes','no','up','down','left','right','on','off','stop','go','silence','unknown']

    def __init__(self, wake_detector=None, data_dir=None, lora_rank=4):
        self.wake_detector = wake_detector
        self.lora_rank = lora_rank
        base_dir = os.path.join(os.path.dirname(__file__), '..', 'data')
        self.data_dir = data_dir or os.path.join(base_dir, 'kws_custom')
        os.makedirs(self.data_dir, exist_ok=True)
        self.samples = {}
        self.custom_labels = []
        self.is_training = False
        self.last_result = None
        self._load_samples()

    def add_sample(self, keyword, audio, sr=16000):
        if len(audio) > sr: audio = audio[:sr]
        elif len(audio) < sr: audio = np.pad(audio, (0, sr - len(audio)))
        if keyword not in self.samples:
            self.samples[keyword] = []
            if keyword not in self.custom_labels: self.custom_labels.append(keyword)
        self.samples[keyword].append(audio.astype(np.float32))
        self._save_samples()
        return {"keyword": keyword, "count": len(self.samples[keyword]),
                "all_counts": self.get_sample_counts(), "status": "saved"}

    def get_sample_counts(self): return {k: len(v) for k, v in self.samples.items()}

    def fine_tune(self, epochs=20, lr=3e-3, neg_ratio=2.0):
        if self.is_training: return {"status": "already_training"}
        for kw, samples in self.samples.items():
            if len(samples) < 5:
                return {"status": "insufficient_data", "keyword": kw, "count": len(samples)}
        self.is_training = True
        t0 = time.time()
        try:
            result = self._do_fine_tune(epochs, lr, neg_ratio)
        except Exception as e:
            self.is_training = False
            import traceback; traceback.print_exc()
            return {"status": "error", "message": str(e)}
        self.is_training = False
        result["time_s"] = round(time.time() - t0, 1)
        self.last_result = result
        return result

    def _do_fine_tune(self, epochs, lr, neg_ratio):
        from nanomamba import create_nc_tcn_20k
        model = create_nc_tcn_20k()
        ckpt_path = os.path.join(PARENT, 'checkpoints', 'best.pt')
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            state = ckpt.get('model_state_dict', ckpt)
            model.load_state_dict(state, strict=False)

        n_base = len(self.BASE_LABELS)
        n_total = n_base + len(self.custom_labels)
        old_head = model.classifier
        d_model = old_head.in_features
        new_head = nn.Linear(d_model, n_total)
        with torch.no_grad():
            new_head.weight[:n_base] = old_head.weight
            new_head.bias[:n_base] = old_head.bias
            nn.init.xavier_uniform_(new_head.weight[n_base:])
            new_head.bias[n_base:].zero_()
        model.classifier = new_head

        lora_modules = []
        for block in model.blocks:
            if hasattr(block, 'in_proj'):
                lora = LoRALinear(block.in_proj, rank=self.lora_rank)
                block.in_proj = lora; lora_modules.append(lora)
            if hasattr(block, 'out_proj'):
                lora = LoRALinear(block.out_proj, rank=self.lora_rank)
                block.out_proj = lora; lora_modules.append(lora)

        for param in model.parameters(): param.requires_grad = False
        for m in lora_modules:
            for param in m.parameters(): param.requires_grad = True
        for param in new_head.parameters(): param.requires_grad = True

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"  [LoRA] Trainable: {trainable:,} / {total_params:,}")

        X, Y = self._prepare_data(neg_ratio)
        model.train()
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        best_loss = float('inf')
        for epoch in range(epochs):
            perm = np.random.permutation(len(X))
            epoch_loss, n_b = 0.0, 0
            for i in range(0, len(X), 8):
                bi = perm[i:i+8]
                bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi])
                by = torch.tensor([Y[j] for j in bi], dtype=torch.long)
                loss = F.cross_entropy(model(bx), by)
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                optimizer.step(); epoch_loss += loss.item(); n_b += 1
            scheduler.step()
            avg = epoch_loss / max(n_b, 1)
            if avg < best_loss: best_loss = avg
            if (epoch+1) % 5 == 0 or epoch == 0:
                print(f"  [LoRA] Epoch {epoch+1}/{epochs} loss={avg:.4f}")

        model.eval()
        save_dir = os.path.join(self.data_dir, 'checkpoints')
        os.makedirs(save_dir, exist_ok=True)
        model_path = os.path.join(save_dir, 'nctcn_custom.pt')
        torch.save({'model_state_dict': model.state_dict(), 'labels': self.BASE_LABELS + self.custom_labels,
                     'n_base_labels': n_base, 'custom_labels': self.custom_labels, 'lora_rank': self.lora_rank}, model_path)

        correct = 0
        with torch.no_grad():
            for j in range(len(X)):
                pred = model(torch.from_numpy(X[j]).float().unsqueeze(0)).argmax(-1).item()
                if pred == Y[j]: correct += 1
        return {"status": "completed", "loss": round(best_loss, 4), "accuracy": round(correct/len(X)*100, 1),
                "samples": len(X), "trainable_params": trainable, "total_params": total_params,
                "labels": self.BASE_LABELS + self.custom_labels, "model_path": model_path}

    def _prepare_data(self, neg_ratio):
        X, Y = [], []
        n_base = len(self.BASE_LABELS)
        for i, kw in enumerate(self.custom_labels):
            li = n_base + i
            for audio in self.samples.get(kw, []):
                X.append(audio); Y.append(li)
                X.append(np.roll(audio, np.random.randint(-1600, 1600))); Y.append(li)
                X.append(audio + np.random.randn(len(audio)).astype(np.float32) * 0.005); Y.append(li)
        n_neg = int(len(X) * neg_ratio)
        si, ui = self.BASE_LABELS.index('silence'), self.BASE_LABELS.index('unknown')
        for _ in range(n_neg // 2):
            X.append(np.random.randn(self.SR).astype(np.float32) * 0.001); Y.append(si)
        for _ in range(n_neg - n_neg // 2):
            X.append(np.random.randn(self.SR).astype(np.float32) * 0.01); Y.append(ui)
        return X, Y

    def _save_samples(self):
        meta = {"custom_labels": self.custom_labels, "samples": {}}
        for kw, audios in self.samples.items():
            kw_dir = os.path.join(self.data_dir, kw)
            os.makedirs(kw_dir, exist_ok=True)
            paths = []
            for i, audio in enumerate(audios):
                path = os.path.join(kw_dir, f"{i:04d}.npy"); np.save(path, audio); paths.append(path)
            meta["samples"][kw] = paths
        with open(os.path.join(self.data_dir, 'meta.json'), 'w') as f: json.dump(meta, f)

    def _load_samples(self):
        mp = os.path.join(self.data_dir, 'meta.json')
        if not os.path.exists(mp): return
        with open(mp) as f: meta = json.load(f)
        self.custom_labels = meta.get('custom_labels', [])
        for kw, paths in meta.get('samples', {}).items():
            self.samples[kw] = [np.load(p) for p in paths if os.path.exists(p)]
'''

with open('core/kws_finetune.py', 'w') as f:
    f.write(KWS_FT_V1.strip())
print("[1/3] kws_finetune.py written (Standard LoRA)")

In [ ]:
# --- NC-OPAL (kws_finetune_ncopal.py) ---
KWS_NCOPAL = r'''
import os, sys, json, time, numpy as np
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F


PARENT = os.path.join(os.path.dirname(__file__), '..')

sys.path.insert(0, PARENT)

class LoRALinear(nn.Module):
    def __init__(self, original, rank=4, alpha=8):
        super().__init__()
        self.original = original
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(original.in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, original.out_features))
    def forward(self, x):
        return self.original(x) + (x @ self.lora_A @ self.lora_B) * self.scaling

class NCOPALFineTuner:
    SR = 16000
    BASE_LABELS = ['yes','no','up','down','left','right','on','off','stop','go','silence','unknown']

    def __init__(self, wake_detector=None, data_dir=None, lora_rank=4):
        self.wake_detector = wake_detector
        self.lora_rank = lora_rank
        base_dir = os.path.join(os.path.dirname(__file__), '..', 'data')
        self.data_dir = data_dir or os.path.join(base_dir, 'kws_custom')
        os.makedirs(self.data_dir, exist_ok=True)
        self.samples = {}
        self.custom_labels = []
        self.is_training = False
        self.last_result = None
        self._load_samples()

    def add_sample(self, keyword, audio, sr=16000):
        if len(audio) > sr: audio = audio[:sr]
        elif len(audio) < sr: audio = np.pad(audio, (0, sr - len(audio)))
        if keyword not in self.samples:
            self.samples[keyword] = []
            if keyword not in self.custom_labels: self.custom_labels.append(keyword)
        self.samples[keyword].append(audio.astype(np.float32))
        self._save_samples()
        return {"keyword": keyword, "count": len(self.samples[keyword]),
                "all_counts": self.get_sample_counts(), "status": "saved"}

    def get_sample_counts(self): return {k: len(v) for k, v in self.samples.items()}

    def fine_tune(self, epochs=20, lr=3e-3, neg_ratio=2.0, lambda_kd=1.0, kd_temperature=4.0):
        if self.is_training: return {"status": "already_training"}
        for kw, samples in self.samples.items():
            if len(samples) < 5:
                return {"status": "insufficient_data", "keyword": kw, "count": len(samples)}
        self.is_training = True
        t0 = time.time()
        try:
            result = self._do_fine_tune(epochs, lr, neg_ratio, lambda_kd, kd_temperature)
        except Exception as e:
            self.is_training = False
            import traceback; traceback.print_exc()
            return {"status": "error", "message": str(e)}
        self.is_training = False
        result["time_s"] = round(time.time() - t0, 1)
        self.last_result = result
        return result

    def _do_fine_tune(self, epochs, lr, neg_ratio, lambda_kd, kd_temperature):
        from nanomamba import create_nc_tcn_20k
        model = create_nc_tcn_20k()
        ckpt_path = os.path.join(PARENT, 'checkpoints', 'best.pt')
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            state = ckpt.get('model_state_dict', ckpt)
            model.load_state_dict(state, strict=False)

        n_base = len(self.BASE_LABELS)
        n_custom = len(self.custom_labels)
        n_total = n_base + n_custom
        d_model = model.classifier.in_features

        # [NOVELTY 1] Prototype-Imprinted Head Init
        print(f"  [NC-OPAL] Prototype-imprinted head init...")
        model.eval()
        emb_hook = {}
        def hook_fn(module, input, output): emb_hook['emb'] = input[0].detach()
        def extract_emb(audio_np):
            h = model.classifier.register_forward_hook(hook_fn)
            with torch.no_grad(): model(torch.from_numpy(audio_np).float().unsqueeze(0))
            h.remove()
            return emb_hook['emb'].squeeze(0)

        custom_protos = {}
        for i, kw in enumerate(self.custom_labels):
            embs = [extract_emb(a) for a in self.samples.get(kw, [])]
            if embs: custom_protos[n_base + i] = torch.stack(embs).mean(dim=0)

        old_head = model.classifier
        new_head = nn.Linear(d_model, n_total)
        with torch.no_grad():
            new_head.weight[:n_base] = old_head.weight
            new_head.bias[:n_base] = old_head.bias
            for i, kw in enumerate(self.custom_labels):
                li = n_base + i
                if li in custom_protos:
                    proto_norm = F.normalize(custom_protos[li], dim=0)
                    scale = old_head.weight.norm(dim=1).mean().item()
                    new_head.weight[li] = proto_norm * scale
                    new_head.bias[li] = old_head.bias.mean().item()
                else:
                    nn.init.xavier_uniform_(new_head.weight[li:li+1])
                    new_head.bias[li] = 0.0
        model.classifier = new_head

        # Apply LoRA
        lora_modules = []
        for block in model.blocks:
            if hasattr(block, 'in_proj'):
                lora = LoRALinear(block.in_proj, rank=self.lora_rank)
                block.in_proj = lora; lora_modules.append(lora)
            if hasattr(block, 'out_proj'):
                lora = LoRALinear(block.out_proj, rank=self.lora_rank)
                block.out_proj = lora; lora_modules.append(lora)

        for param in model.parameters(): param.requires_grad = False
        for m in lora_modules:
            for param in m.parameters(): param.requires_grad = True
        for param in new_head.parameters(): param.requires_grad = True

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"  [NC-OPAL] Trainable: {trainable:,} / {total_params:,}")

        # [NOVELTY 2] Frozen teacher for KD
        teacher = None
        if lambda_kd > 0:
            teacher = create_nc_tcn_20k()
            if os.path.exists(ckpt_path):
                teacher.load_state_dict(state, strict=False)
            teacher.eval()
            for p in teacher.parameters(): p.requires_grad = False
            print(f"  [NC-OPAL] KD teacher loaded (lambda={lambda_kd}, T={kd_temperature})")

        X, Y = self._prepare_data(neg_ratio)
        print(f"  [NC-OPAL] Training data: {len(X)} samples")

        model.train()
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        best_loss = float('inf')

        for epoch in range(epochs):
            perm = np.random.permutation(len(X))
            epoch_loss, epoch_kd, n_b = 0.0, 0.0, 0
            for i in range(0, len(X), 8):
                bi = perm[i:i+8]
                bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi])
                by = torch.tensor([Y[j] for j in bi], dtype=torch.long)
                logits = model(bx)
                cls_loss = F.cross_entropy(logits, by)

                kd_loss = torch.tensor(0.0)
                if teacher is not None and lambda_kd > 0:
                    base_mask = by < n_base
                    if base_mask.any():
                        with torch.no_grad(): t_logits = teacher(bx[base_mask])
                        s_base = logits[base_mask][:, :n_base] / kd_temperature
                        t_base = t_logits / kd_temperature
                        kd_loss = F.kl_div(F.log_softmax(s_base, dim=-1),
                                          F.softmax(t_base, dim=-1),
                                          reduction='batchmean') * (kd_temperature ** 2)

                total_loss = cls_loss + lambda_kd * kd_loss
                optimizer.zero_grad(); total_loss.backward()
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                optimizer.step()
                epoch_loss += cls_loss.item(); epoch_kd += kd_loss.item(); n_b += 1

            scheduler.step()
            avg = epoch_loss / max(n_b, 1)
            if avg < best_loss: best_loss = avg
            if (epoch+1) % 5 == 0 or epoch == 0:
                print(f"  [NC-OPAL] Epoch {epoch+1}/{epochs} cls={avg:.4f} kd={epoch_kd/max(n_b,1):.4f}")

        model.eval()
        save_dir = os.path.join(self.data_dir, 'checkpoints')
        os.makedirs(save_dir, exist_ok=True)
        model_path = os.path.join(save_dir, 'nctcn_ncopal.pt')
        torch.save({'model_state_dict': model.state_dict(), 'labels': self.BASE_LABELS + self.custom_labels,
                     'n_base_labels': n_base, 'custom_labels': self.custom_labels,
                     'lora_rank': self.lora_rank, 'algorithm': 'NC-OPAL'}, model_path)

        correct = 0
        per_class = defaultdict(lambda: [0,0])
        with torch.no_grad():
            for j in range(len(X)):
                pred = model(torch.from_numpy(X[j]).float().unsqueeze(0)).argmax(-1).item()
                per_class[Y[j]][1] += 1
                if pred == Y[j]: correct += 1; per_class[Y[j]][0] += 1

        acc = correct / max(len(X), 1) * 100
        all_labels = self.BASE_LABELS + self.custom_labels
        class_accs = {}
        for idx, name in enumerate(all_labels):
            c, t = per_class[idx]
            if t > 0: class_accs[name] = round(c/t*100, 1)
        base_c = sum(per_class[i][0] for i in range(n_base))
        base_t = sum(per_class[i][1] for i in range(n_base))
        base_acc = base_c / max(base_t, 1) * 100
        new_c = sum(per_class[i][0] for i in range(n_base, n_total))
        new_t = sum(per_class[i][1] for i in range(n_base, n_total))
        new_acc = new_c / max(new_t, 1) * 100
        print(f"  [NC-OPAL] Acc: {acc:.1f}% (base: {base_acc:.1f}%, new: {new_acc:.1f}%)")

        return {"status": "completed", "algorithm": "NC-OPAL", "loss": round(best_loss, 4),
                "accuracy": round(acc, 1), "base_accuracy": round(base_acc, 1),
                "new_accuracy": round(new_acc, 1), "forgetting": round(100-base_acc, 1),
                "samples": len(X), "trainable_params": trainable, "total_params": total_params,
                "class_accuracies": class_accs, "labels": self.BASE_LABELS + self.custom_labels,
                "model_path": model_path}

    def _prepare_data(self, neg_ratio):
        X, Y = [], []
        n_base = len(self.BASE_LABELS)
        for i, kw in enumerate(self.custom_labels):
            li = n_base + i
            for audio in self.samples.get(kw, []):
                X.append(audio); Y.append(li)
                X.append(np.roll(audio, np.random.randint(-1600, 1600))); Y.append(li)
                X.append(audio + np.random.randn(len(audio)).astype(np.float32) * 0.005); Y.append(li)
        n_neg = int(len(X) * neg_ratio)
        si, ui = self.BASE_LABELS.index('silence'), self.BASE_LABELS.index('unknown')
        for _ in range(n_neg // 2):
            X.append(np.random.randn(self.SR).astype(np.float32) * 0.001); Y.append(si)
        for _ in range(n_neg - n_neg // 2):
            X.append(np.random.randn(self.SR).astype(np.float32) * 0.01); Y.append(ui)
        return X, Y

    def _save_samples(self):
        meta = {"custom_labels": self.custom_labels, "samples": {}}
        for kw, audios in self.samples.items():
            kw_dir = os.path.join(self.data_dir, kw)
            os.makedirs(kw_dir, exist_ok=True)
            paths = []
            for i, audio in enumerate(audios):
                path = os.path.join(kw_dir, f"{i:04d}.npy"); np.save(path, audio); paths.append(path)
            meta["samples"][kw] = paths
        with open(os.path.join(self.data_dir, 'meta.json'), 'w') as f: json.dump(meta, f)

    def _load_samples(self):
        mp = os.path.join(self.data_dir, 'meta.json')
        if not os.path.exists(mp): return
        with open(mp) as f: meta = json.load(f)
        self.custom_labels = meta.get('custom_labels', [])
        for kw, paths in meta.get('samples', {}).items():
            self.samples[kw] = [np.load(p) for p in paths if os.path.exists(p)]
'''

with open('core/kws_finetune_ncopal.py', 'w') as f:
    f.write(KWS_NCOPAL.strip())
print("[2/3] kws_finetune_ncopal.py written (NC-OPAL)")

In [ ]:
# --- NC-ALoRA-PR (kws_finetune_v2.py) ---
KWS_V2 = r'''
import os, sys, json, time, math, numpy as np
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F


PARENT = os.path.join(os.path.dirname(__file__), '..')

sys.path.insert(0, PARENT)

class AdaptiveLoRALinear(nn.Module):
    def __init__(self, original, max_rank=8, min_rank=1, alpha=16.0):
        super().__init__()
        self.original = original
        self.max_rank = max_rank
        self.min_rank = min_rank
        self.alpha = alpha
        self.lora_A = nn.Parameter(torch.randn(original.in_features, max_rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(max_rank, original.out_features))
        self.rank_importance = nn.Parameter(torch.ones(max_rank))
        self._effective_rank = max_rank
        self._mask = None

    @property
    def scaling(self): return self.alpha / self._effective_rank

    def forward(self, x):
        base_out = self.original(x)
        gate = torch.sigmoid(self.rank_importance)
        if self._mask is not None: gate = gate * self._mask
        return base_out + (x @ (self.lora_A * gate.unsqueeze(0)) @ self.lora_B) * self.scaling

    def prune_rank(self, threshold=0.1):
        with torch.no_grad():
            imp = torch.sigmoid(self.rank_importance).detach()
            mask = (imp > threshold).float()
            if mask.sum() < self.min_rank:
                _, topk = imp.topk(self.min_rank)
                mask = torch.zeros_like(mask); mask[topk] = 1.0
            self._mask = mask
            self._effective_rank = max(int(mask.sum().item()), self.min_rank)
        return self._effective_rank

    def get_effective_rank(self): return self._effective_rank

class PrototypeMemory:
    def __init__(self, n_classes, d_model):
        self.n_classes = n_classes
        self.d_model = d_model
        self.prototypes = torch.zeros(n_classes, d_model)
        self.counts = torch.zeros(n_classes)
        self._finalized = False

    def update(self, embeddings, labels):
        for cls in range(self.n_classes):
            mask = labels == cls
            if mask.any():
                self.prototypes[cls] += embeddings[mask].sum(0).detach()
                self.counts[cls] += mask.sum().item()

    def finalize(self):
        for cls in range(self.n_classes):
            if self.counts[cls] > 0: self.prototypes[cls] /= self.counts[cls]
        norms = self.prototypes.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.prototypes = self.prototypes / norms
        self._finalized = True

    def distillation_loss(self, emb, labels, base_classes):
        if not self._finalized: return torch.tensor(0.0)
        loss, n = torch.tensor(0.0), 0
        for cls in range(base_classes):
            mask = labels == cls
            if mask.any() and self.counts[cls] > 0:
                cur = F.normalize(emb[mask], dim=1)
                sim = (cur * self.prototypes[cls].unsqueeze(0)).sum(1)
                loss = loss + (1-sim).mean(); n += 1
        return loss / max(n, 1)

    def contrastive_loss(self, emb, labels, base_classes, margin=0.5):
        if not self._finalized: return torch.tensor(0.0)
        loss, n = torch.tensor(0.0), 0
        for cls in range(base_classes, self.n_classes):
            mask = labels == cls
            if not mask.any(): continue
            cur = F.normalize(emb[mask], dim=1)
            own = self.prototypes[cls].unsqueeze(0) if self.counts[cls] > 0 else cur.mean(0, keepdim=True)
            pos_dist = 1 - (cur * own).sum(1)
            neg_sim = cur @ self.prototypes[:base_classes].t()
            neg_dist = 1 - neg_sim.max(1).values
            loss = loss + F.relu(pos_dist - neg_dist + margin).mean(); n += 1
        return loss / max(n, 1)

class SpectralAugCurriculum:
    def __init__(self, total_epochs, sr=16000):
        self.total_epochs = total_epochs; self.sr = sr
    def augment(self, audio, epoch):
        p = epoch / max(self.total_epochs - 1, 1)
        aug = audio.copy()
        aug = np.roll(aug, int(np.random.uniform(-0.05, 0.05) * self.sr * (0.5 + p)))
        if p > 0.2:
            aug += np.random.randn(len(aug)).astype(np.float32) * np.interp(p, [0.2,1.0], [0.002,0.03])
        if p > 0.5:
            spec = np.fft.rfft(aug); n_bins = len(spec)
            mw = int(n_bins * np.interp(p, [0.5,1.0], [0.05,0.2]))
            if mw > 0:
                st = np.random.randint(0, max(n_bins - mw, 1))
                spec[st:st+mw] = 0
            aug = np.fft.irfft(spec, n=len(audio)).astype(np.float32)
        pk = np.abs(aug).max()
        if pk > 0: aug = aug / pk * min(np.abs(audio).max(), 0.95)
        if len(aug) > self.sr: aug = aug[:self.sr]
        elif len(aug) < self.sr: aug = np.pad(aug, (0, self.sr - len(aug)))
        return aug.astype(np.float32)

class GradientSNRMonitor:
    def __init__(self):
        self.grad_history = defaultdict(list); self.window_size = 5
    def record(self, name, grad):
        self.grad_history[name].append(grad.detach().clone())
        if len(self.grad_history[name]) > self.window_size: self.grad_history[name].pop(0)
    def compute_snr(self, name):
        grads = self.grad_history.get(name, [])
        if len(grads) < 2: return 1.0
        st = torch.stack(grads)
        var = st.var(0).mean(); sig = (st.mean(0)**2).mean()
        return (sig / (var + 1e-10)).item() if var > 1e-10 else 10.0
    def get_layer_importance(self): return {n: self.compute_snr(n) for n in self.grad_history}

class NCALoRAPRFineTuner:
    SR = 16000
    BASE_LABELS = ['yes','no','up','down','left','right','on','off','stop','go','silence','unknown']

    def __init__(self, wake_detector=None, data_dir=None, max_rank=8, min_rank=1):
        self.wake_detector = wake_detector
        self.max_rank = max_rank; self.min_rank = min_rank
        base_dir = os.path.join(os.path.dirname(__file__), '..', 'data')
        self.data_dir = data_dir or os.path.join(base_dir, 'kws_custom')
        os.makedirs(self.data_dir, exist_ok=True)
        self.samples = {}; self.custom_labels = []
        self.is_training = False; self.last_result = None
        self._load_samples()

    def add_sample(self, keyword, audio, sr=16000):
        if len(audio) > sr: audio = audio[:sr]
        elif len(audio) < sr: audio = np.pad(audio, (0, sr - len(audio)))
        if keyword not in self.samples:
            self.samples[keyword] = []
            if keyword not in self.custom_labels: self.custom_labels.append(keyword)
        self.samples[keyword].append(audio.astype(np.float32))
        self._save_samples()
        return {"keyword": keyword, "count": len(self.samples[keyword]),
                "all_counts": self.get_sample_counts(), "status": "saved"}

    def get_sample_counts(self): return {k: len(v) for k, v in self.samples.items()}

    def fine_tune(self, epochs=30, lr=3e-3, neg_ratio=2.0, lambda_proto=0.3,
                  lambda_contrast=0.2, prune_after_epoch=10, margin=0.5):
        if self.is_training: return {"status": "already_training"}
        for kw, s in self.samples.items():
            if len(s) < 3: return {"status": "insufficient_data", "keyword": kw, "count": len(s)}
        self.is_training = True; t0 = time.time()
        try:
            result = self._do_fine_tune(epochs, lr, neg_ratio, lambda_proto, lambda_contrast, prune_after_epoch, margin)
        except Exception as e:
            self.is_training = False
            import traceback; traceback.print_exc()
            return {"status": "error", "message": str(e)}
        self.is_training = False
        result["time_s"] = round(time.time() - t0, 1)
        self.last_result = result
        return result

    def _forward_with_emb(self, model, audio, d_model):
        emb_dict = {}
        def hook(m, inp, out): emb_dict['e'] = inp[0]
        h = model.classifier.register_forward_hook(hook)
        logits = model(audio); h.remove()
        return logits, emb_dict.get('e', torch.zeros(audio.shape[0], d_model))

    def _do_fine_tune(self, epochs, lr, neg_ratio, lp, lc, prune_ep, margin):
        from nanomamba import create_nc_tcn_20k
        model = create_nc_tcn_20k()
        ckpt_path = os.path.join(PARENT, 'checkpoints', 'best.pt')
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            state = ckpt.get('model_state_dict', ckpt)
            model.load_state_dict(state, strict=False)

        n_base = len(self.BASE_LABELS)
        n_total = n_base + len(self.custom_labels)
        old_head = model.classifier; d_model = old_head.in_features
        new_head = nn.Linear(d_model, n_total)
        with torch.no_grad():
            new_head.weight[:n_base] = old_head.weight; new_head.bias[:n_base] = old_head.bias
            nn.init.xavier_uniform_(new_head.weight[n_base:]); new_head.bias[n_base:].zero_()
        model.classifier = new_head

        alora_mods, alora_names = [], []
        for bi, block in enumerate(model.blocks):
            if hasattr(block, 'in_proj'):
                a = AdaptiveLoRALinear(block.in_proj, max_rank=self.max_rank, min_rank=self.min_rank, alpha=self.max_rank*2)
                block.in_proj = a; alora_mods.append(a); alora_names.append(f"b{bi}.in")
            if hasattr(block, 'out_proj'):
                a = AdaptiveLoRALinear(block.out_proj, max_rank=self.max_rank, min_rank=self.min_rank, alpha=self.max_rank*2)
                block.out_proj = a; alora_mods.append(a); alora_names.append(f"b{bi}.out")

        for p in model.parameters(): p.requires_grad = False
        for m in alora_mods:
            for p in m.parameters(): p.requires_grad = True
        for p in new_head.parameters(): p.requires_grad = True

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"  [NC-ALoRA-PR] Trainable: {trainable:,}/{total_params:,}, ALoRA: {len(alora_mods)}")

        proto_mem = PrototypeMemory(n_total, d_model)
        model.eval()
        X_proto, Y_proto = [], []
        for _ in range(30):
            X_proto.append(np.random.randn(self.SR).astype(np.float32)*0.001); Y_proto.append(self.BASE_LABELS.index('silence'))
            X_proto.append(np.random.randn(self.SR).astype(np.float32)*0.01); Y_proto.append(self.BASE_LABELS.index('unknown'))
        for i, kw in enumerate(self.custom_labels):
            for a in self.samples.get(kw, []): X_proto.append(a); Y_proto.append(n_base+i)
        with torch.no_grad():
            for j in range(0, len(X_proto), 16):
                bx = torch.stack([torch.from_numpy(X_proto[k]).float() for k in range(j, min(j+16, len(X_proto)))])
                by = torch.tensor(Y_proto[j:j+16], dtype=torch.long)
                _, emb = self._forward_with_emb(model, bx, d_model)
                proto_mem.update(emb, by)
        proto_mem.finalize()

        X, Y = self._prepare_data(neg_ratio)
        aug = SpectralAugCurriculum(epochs, self.SR)
        gmon = GradientSNRMonitor()
        warmup_ep = min(5, epochs//4)

        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs,
                                                        pct_start=warmup_ep/epochs, anneal_strategy='cos')
        model.train(); best_loss = float('inf')
        for epoch in range(epochs):
            perm = np.random.permutation(len(X))
            el, ecl, epl, ecol, nb = 0,0,0,0,0
            for i in range(0, len(X), 8):
                bi = perm[i:i+8]
                bx = torch.stack([torch.from_numpy(aug.augment(X[j], epoch)).float() for j in bi])
                by = torch.tensor([Y[j] for j in bi], dtype=torch.long)
                logits, emb = self._forward_with_emb(model, bx, d_model)
                cls = F.cross_entropy(logits, by)
                proto = proto_mem.distillation_loss(emb, by, n_base)
                contr = proto_mem.contrastive_loss(emb, by, n_base, margin)
                pw = lp * min(1.0, epoch / max(warmup_ep, 1))
                total = cls + pw * proto + lc * contr
                optimizer.zero_grad(); total.backward()
                for n, am in zip(alora_names, alora_mods):
                    if am.lora_A.grad is not None: gmon.record(n, am.lora_A.grad)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                optimizer.step()
                el += total.item(); ecl += cls.item(); epl += proto.item(); ecol += contr.item(); nb += 1
            scheduler.step()
            avg = el / max(nb, 1)
            if avg < best_loss: best_loss = avg
            if epoch == prune_ep:
                snrs = gmon.get_layer_importance()
                for n, am in zip(alora_names, alora_mods):
                    snr = snrs.get(n, 1.0)
                    th = 0.3 if snr < 0.5 else 0.15 if snr < 1.0 else 0.05
                    er = am.prune_rank(th)
                    print(f"    {n}: SNR={snr:.2f} -> rank={er}")
            if (epoch+1) % 5 == 0:
                model.eval()
                for i, kw in enumerate(self.custom_labels):
                    audios = self.samples.get(kw, [])
                    if audios:
                        with torch.no_grad():
                            b = torch.stack([torch.from_numpy(a).float() for a in audios[:16]])
                            _, e = self._forward_with_emb(model, b, d_model)
                            proto_mem.prototypes[n_base+i] = F.normalize(e.mean(0), dim=0)
                model.train()
            if (epoch+1) % 5 == 0 or epoch == 0:
                print(f"  [NC-ALoRA-PR] Ep {epoch+1}/{epochs} loss={avg:.4f} cls={ecl/max(nb,1):.4f} proto={epl/max(nb,1):.4f}")

        model.eval()
        save_dir = os.path.join(self.data_dir, 'checkpoints'); os.makedirs(save_dir, exist_ok=True)
        model_path = os.path.join(save_dir, 'nctcn_ncalora_pr.pt')
        final_ranks = {n: am.get_effective_rank() for n, am in zip(alora_names, alora_mods)}
        torch.save({'model_state_dict': model.state_dict(), 'labels': self.BASE_LABELS + self.custom_labels,
                     'n_base_labels': n_base, 'custom_labels': self.custom_labels,
                     'max_rank': self.max_rank, 'effective_ranks': final_ranks, 'algorithm': 'NC-ALoRA-PR'}, model_path)

        correct = 0; pc_c, pc_t = defaultdict(int), defaultdict(int)
        with torch.no_grad():
            for j in range(len(X)):
                pred = model(torch.from_numpy(X[j]).float().unsqueeze(0)).argmax(-1).item()
                pc_t[Y[j]] += 1
                if pred == Y[j]: correct += 1; pc_c[Y[j]] += 1
        acc = correct / len(X) * 100
        all_labels = self.BASE_LABELS + self.custom_labels
        class_accs = {all_labels[i]: round(pc_c[i]/pc_t[i]*100, 1) for i in range(len(all_labels)) if pc_t[i] > 0}
        ba = sum(pc_c[i] for i in range(n_base)) / max(sum(pc_t[i] for i in range(n_base)), 1) * 100
        na = sum(pc_c[i] for i in range(n_base, n_total)) / max(sum(pc_t[i] for i in range(n_base, n_total)), 1) * 100
        print(f"  [NC-ALoRA-PR] Acc: {acc:.1f}% base: {ba:.1f}% new: {na:.1f}% ranks: {final_ranks}")

        return {"status": "completed", "algorithm": "NC-ALoRA-PR", "loss": round(best_loss, 4),
                "accuracy": round(acc, 1), "base_accuracy": round(ba, 1), "new_accuracy": round(na, 1),
                "forgetting": round(100-ba, 1), "samples": len(X), "trainable_params": trainable,
                "total_params": total_params, "effective_ranks": final_ranks, "class_accuracies": class_accs,
                "labels": self.BASE_LABELS + self.custom_labels, "model_path": model_path}

    def _prepare_data(self, neg_ratio):
        X, Y = [], []; n_base = len(self.BASE_LABELS)
        for i, kw in enumerate(self.custom_labels):
            for a in self.samples.get(kw, []): X.append(a); Y.append(n_base+i)
        n_neg = int(len(X) * neg_ratio)
        si, ui = self.BASE_LABELS.index('silence'), self.BASE_LABELS.index('unknown')
        for _ in range(n_neg // 2):
            X.append(np.random.randn(self.SR).astype(np.float32)*0.001); Y.append(si)
        for _ in range(n_neg - n_neg // 2):
            X.append(np.random.randn(self.SR).astype(np.float32)*0.01); Y.append(ui)
        return X, Y

    def _save_samples(self):
        meta = {"custom_labels": self.custom_labels, "samples": {}}
        for kw, audios in self.samples.items():
            kw_dir = os.path.join(self.data_dir, kw); os.makedirs(kw_dir, exist_ok=True)
            paths = []
            for i, a in enumerate(audios):
                p = os.path.join(kw_dir, f"{i:04d}.npy"); np.save(p, a); paths.append(p)
            meta["samples"][kw] = paths
        with open(os.path.join(self.data_dir, 'meta.json'), 'w') as f: json.dump(meta, f)

    def _load_samples(self):
        mp = os.path.join(self.data_dir, 'meta.json')
        if not os.path.exists(mp): return
        with open(mp) as f: meta = json.load(f)
        self.custom_labels = meta.get('custom_labels', [])
        for kw, paths in meta.get('samples', {}).items():
            self.samples[kw] = [np.load(p) for p in paths if os.path.exists(p)]
'''

with open('core/kws_finetune_v2.py', 'w') as f:
    f.write(KWS_V2.strip())
print("[3/3] kws_finetune_v2.py written (NC-ALoRA-PR)")
print("\nAll fine-tuning modules ready!")

In [ ]:
# ============================================================
# 4. Config & imports
# ============================================================
import sys, os

# Ensure paths (redundant safety for Colab)
REPO_ROOT = '/content/NC-KWS-FineTuning'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

import numpy as np
import torch
import torchaudio
import time
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)

SR = 16000
BASE_LABELS = ['yes', 'no', 'up', 'down', 'left', 'right',
               'on', 'off', 'stop', 'go', 'silence', 'unknown']
NEW_KEYWORDS = ['marvin', 'sheila', 'bed']
N_TRAIN = 20   # samples per new keyword for training
N_TEST = 100   # samples per class for evaluation

print(f'Base classes: {BASE_LABELS}')
print(f'New keywords: {NEW_KEYWORDS}')
print(f'Train: {N_TRAIN}/kw, Test: {N_TEST}/class')

In [ ]:
# ============================================================
# 5. Load audio data from GSC V2
# ============================================================
def load_wav_samples(keyword, n, offset=0):
    """Load n wav files for a keyword from GSC."""
    kw_dir = os.path.join(GSC_DIR, keyword)
    if not os.path.exists(kw_dir):
        print(f'  WARNING: {kw_dir} not found')
        return []
    files = sorted([f for f in os.listdir(kw_dir) if f.endswith('.wav')])
    samples = []
    for f in files[offset:offset + n]:
        try:
            waveform, sr = torchaudio.load(os.path.join(kw_dir, f))
            audio = waveform[0].numpy()
            if len(audio) > SR: audio = audio[:SR]
            elif len(audio) < SR: audio = np.pad(audio, (0, SR - len(audio)))
            samples.append(audio.astype(np.float32))
        except:
            continue
    return samples

# Load training data (new keywords only)
print('Loading training data...')
train_data = {}
for kw in NEW_KEYWORDS:
    train_data[kw] = load_wav_samples(kw, N_TRAIN)
    print(f'  {kw}: {len(train_data[kw])} samples')

# Load test data (all classes, held-out from training)
print('\nLoading test data...')
test_data = {}
for kw in NEW_KEYWORDS:
    test_data[kw] = load_wav_samples(kw, N_TEST, offset=N_TRAIN + 200)
    print(f'  {kw}: {len(test_data[kw])} test')

for kw in ['yes', 'no', 'up', 'down', 'left', 'right',
           'on', 'off', 'stop', 'go']:
    test_data[kw] = load_wav_samples(kw, N_TEST, offset=500)
    print(f'  {kw}: {len(test_data[kw])} test')

test_data['silence'] = [np.random.randn(SR).astype(np.float32) * 0.002
                        for _ in range(N_TEST)]
print(f'  silence: {N_TEST} test')

# Unknown = words NOT in base or new
unknown_words = ['bird', 'cat', 'dog', 'happy', 'house', 'tree', 'wow']
unknown_samples = []
for w in unknown_words:
    unknown_samples.extend(load_wav_samples(w, N_TEST // len(unknown_words), offset=300))
test_data['unknown'] = unknown_samples[:N_TEST]
print(f'  unknown: {len(test_data["unknown"])} test')

total_test = sum(len(v) for v in test_data.values())
print(f'\nTotal: {sum(len(v) for v in train_data.values())} train, {total_test} test samples')

## 6. Load Base Model & Baseline Evaluation

In [ ]:
import sys, os
if '/content/NC-KWS-FineTuning' not in sys.path:
    sys.path.insert(0, '/content/NC-KWS-FineTuning')

from nanomamba import create_nc_tcn_20k

def load_base_model():
    model = create_nc_tcn_20k()
    ckpt_path = 'checkpoints/best.pt'
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        state = ckpt.get('model_state_dict', ckpt)
        model.load_state_dict(state, strict=False)
    model.eval()
    return model

def evaluate(model, labels, test_data, device='cpu'):
    """Evaluate model on test data."""
    model = model.to(device).eval()
    results = {}
    total_c, total_n = 0, 0
    
    with torch.no_grad():
        for kw, samples in test_data.items():
            true_idx = labels.index(kw) if kw in labels else labels.index('unknown')
            correct = 0
            for audio in samples:
                x = torch.from_numpy(audio).float().unsqueeze(0).to(device)
                pred = model(x).argmax(dim=-1).item()
                if pred == true_idx:
                    correct += 1
            results[kw] = correct / max(len(samples), 1) * 100
            total_c += correct
            total_n += len(samples)
    
    results['_overall'] = total_c / max(total_n, 1) * 100
    return results

# Baseline
print('Evaluating base model (no fine-tuning)...')
base_model = load_base_model()
base_eval = evaluate(base_model, BASE_LABELS, test_data)

print(f"\n{'Keyword':<12} {'Accuracy':>8}")
print('-' * 22)
for kw in NEW_KEYWORDS + BASE_LABELS[:10] + ['silence', 'unknown']:
    marker = ' *NEW*' if kw in NEW_KEYWORDS else ''
    print(f"{kw:<12} {base_eval.get(kw, 0):>7.1f}%{marker}")
print(f"{'OVERALL':<12} {base_eval['_overall']:>7.1f}%")

## 7. Fine-Tuning: Method 1 - Standard LoRA (Baseline)

In [ ]:
import tempfile
from core.kws_finetune import KWSFineTuner, LoRALinear

d1 = os.path.join(tempfile.gettempdir(), 'kws_real_lora')
ft1 = KWSFineTuner(data_dir=d1)
for kw in NEW_KEYWORDS:
    for audio in train_data[kw]:
        ft1.add_sample(kw, audio)

print(f'Samples: {ft1.get_sample_counts()}')
t0 = time.time()
r1 = ft1.fine_tune(epochs=20, lr=3e-3)
print(f"\nResult: {r1['status']}, train_acc={r1.get('accuracy')}%, time={time.time()-t0:.1f}s")

In [ ]:
# Evaluate Standard LoRA
if r1.get('status') == 'completed':
    model1 = create_nc_tcn_20k()
    ft_ckpt = torch.load(r1['model_path'], map_location='cpu', weights_only=False)
    labels1 = ft_ckpt['labels']
    d = model1.classifier.in_features
    model1.classifier = torch.nn.Linear(d, len(labels1))
    for block in model1.blocks:
        if hasattr(block, 'in_proj'):
            block.in_proj = LoRALinear(block.in_proj, rank=4)
        if hasattr(block, 'out_proj'):
            block.out_proj = LoRALinear(block.out_proj, rank=4)
    model1.load_state_dict(ft_ckpt['model_state_dict'], strict=False)
    
    eval1 = evaluate(model1, labels1, test_data)
    print(f"Standard LoRA test results:")
    for kw in NEW_KEYWORDS + BASE_LABELS[:10] + ['silence', 'unknown']:
        marker = ' *NEW*' if kw in NEW_KEYWORDS else ''
        print(f"  {kw:<12} {eval1.get(kw, 0):>7.1f}%{marker}")
    print(f"  {'OVERALL':<12} {eval1['_overall']:>7.1f}%")

## 8. Fine-Tuning: Method 2 - NC-ALoRA-PR (Novel)

In [ ]:
from core.kws_finetune_v2 import NCALoRAPRFineTuner, AdaptiveLoRALinear

d2 = os.path.join(tempfile.gettempdir(), 'kws_real_alora')
ft2 = NCALoRAPRFineTuner(data_dir=d2)
for kw in NEW_KEYWORDS:
    for audio in train_data[kw]:
        ft2.add_sample(kw, audio)

t0 = time.time()
r2 = ft2.fine_tune(epochs=25, lr=3e-3)
print(f"\nResult: {r2['status']}, train_acc={r2.get('accuracy')}%, time={time.time()-t0:.1f}s")

In [ ]:
# Evaluate NC-ALoRA-PR
if r2.get('status') == 'completed':
    model2 = create_nc_tcn_20k()
    ft_ckpt = torch.load(r2['model_path'], map_location='cpu', weights_only=False)
    labels2 = ft_ckpt['labels']
    d = model2.classifier.in_features
    model2.classifier = torch.nn.Linear(d, len(labels2))
    for block in model2.blocks:
        if hasattr(block, 'in_proj'):
            block.in_proj = AdaptiveLoRALinear(block.in_proj, max_rank=8)
        if hasattr(block, 'out_proj'):
            block.out_proj = AdaptiveLoRALinear(block.out_proj, max_rank=8)
    model2.load_state_dict(ft_ckpt['model_state_dict'], strict=False)
    
    eval2 = evaluate(model2, labels2, test_data)
    print(f"NC-ALoRA-PR test results:")
    for kw in NEW_KEYWORDS + BASE_LABELS[:10] + ['silence', 'unknown']:
        marker = ' *NEW*' if kw in NEW_KEYWORDS else ''
        print(f"  {kw:<12} {eval2.get(kw, 0):>7.1f}%{marker}")
    print(f"  {'OVERALL':<12} {eval2['_overall']:>7.1f}%")

## 9. Fine-Tuning: Method 3 - NC-OPAL (Novel)

In [ ]:
from core.kws_finetune_ncopal import NCOPALFineTuner
from core.kws_finetune_ncopal import LoRALinear as OPALLoRALinear

d3 = os.path.join(tempfile.gettempdir(), 'kws_real_ncopal')
ft3 = NCOPALFineTuner(data_dir=d3)
for kw in NEW_KEYWORDS:
    for audio in train_data[kw]:
        ft3.add_sample(kw, audio)

t0 = time.time()
r3 = ft3.fine_tune(epochs=20, lr=3e-3, lambda_kd=1.0)
print(f"\nResult: {r3['status']}, train_acc={r3.get('accuracy')}%, time={time.time()-t0:.1f}s")

In [ ]:
# Evaluate NC-OPAL
if r3.get('status') == 'completed':
    model3 = create_nc_tcn_20k()
    ft_ckpt = torch.load(r3['model_path'], map_location='cpu', weights_only=False)
    labels3 = ft_ckpt['labels']
    d = model3.classifier.in_features
    model3.classifier = torch.nn.Linear(d, len(labels3))
    for block in model3.blocks:
        if hasattr(block, 'in_proj'):
            block.in_proj = OPALLoRALinear(block.in_proj, rank=4)
        if hasattr(block, 'out_proj'):
            block.out_proj = OPALLoRALinear(block.out_proj, rank=4)
    model3.load_state_dict(ft_ckpt['model_state_dict'], strict=False)
    
    eval3 = evaluate(model3, labels3, test_data)
    print(f"NC-OPAL test results:")
    for kw in NEW_KEYWORDS + BASE_LABELS[:10] + ['silence', 'unknown']:
        marker = ' *NEW*' if kw in NEW_KEYWORDS else ''
        print(f"  {kw:<12} {eval3.get(kw, 0):>7.1f}%{marker}")
    print(f"  {'OVERALL':<12} {eval3['_overall']:>7.1f}%")

## 10. Final Comparison Table

In [ ]:
import pandas as pd

all_evals = {
    'Base (no FT)': base_eval,
}
if 'eval1' in dir(): all_evals['Standard LoRA'] = eval1
if 'eval2' in dir(): all_evals['NC-ALoRA-PR'] = eval2
if 'eval3' in dir(): all_evals['NC-OPAL'] = eval3

# Build comparison table
rows = []
for kw in NEW_KEYWORDS + BASE_LABELS[:10] + ['silence', 'unknown']:
    row = {'Keyword': kw + (' *NEW*' if kw in NEW_KEYWORDS else '')}
    for method, ev in all_evals.items():
        row[method] = f"{ev.get(kw, 0):.1f}%"
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Summary metrics
print(f"\n{'='*70}")
print(f"{'Method':<18} {'Overall':>8} {'NewKW':>8} {'BaseKW':>8} {'Forgetting':>10}")
print(f"{'-'*52}")

base_kws = BASE_LABELS[:10] + ['silence', 'unknown']
base_baseline = np.mean([base_eval.get(k, 0) for k in base_kws])

for method, ev in all_evals.items():
    overall = ev['_overall']
    new_acc = np.mean([ev.get(k, 0) for k in NEW_KEYWORDS])
    base_acc = np.mean([ev.get(k, 0) for k in base_kws])
    forget = base_baseline - base_acc
    print(f"{method:<18} {overall:>7.1f}% {new_acc:>7.1f}% {base_acc:>7.1f}% {forget:>+9.1f}%")

## 11. Few-Shot Ablation (3, 5, 10, 20 shot)

In [ ]:
# Few-Shot Ablation: LoRA vs NC-OPAL across different shot counts
shot_results = {}
shot_evals = {}

for n_shot in [3, 5, 10, 20]:
    print(f"\n{'='*50}")
    print(f"  {n_shot}-shot fine-tuning")
    print(f"{'='*50}")

    for method_name, FTClass in [
        ('LoRA', KWSFineTuner),
        ('NC-OPAL', NCOPALFineTuner),
    ]:
        import tempfile
        d = os.path.join(tempfile.gettempdir(), f'kws_shot_{method_name}_{n_shot}')
        ft = FTClass(data_dir=d)
        for kw in NEW_KEYWORDS:
            for audio in train_data[kw][:n_shot]:
                ft.add_sample(kw, audio)

        kwargs = {'epochs': 20, 'lr': 3e-3}
        if method_name == 'NC-OPAL':
            kwargs['lambda_kd'] = 1.0

        r = ft.fine_tune(**kwargs)

        if r.get('status') == 'completed':
            key = (method_name, n_shot)
            shot_results[key] = r.get('accuracy', 0)
            print(f"  {method_name} {n_shot}-shot: train_acc={r.get('accuracy')}%")

            # Also eval on test set for this shot count
            from core.kws_finetune import LoRALinear as LR
            from core.kws_finetune_ncopal import LoRALinear as OLR
            from nanomamba import create_nc_tcn_20k
            import torch.nn

            m = create_nc_tcn_20k()
            fc = torch.load(r['model_path'], map_location='cpu', weights_only=False)
            lbls = fc['labels']
            dm = m.classifier.in_features
            m.classifier = torch.nn.Linear(dm, len(lbls))
            LClass = OLR if method_name == 'NC-OPAL' else LR
            for block in m.blocks:
                if hasattr(block, 'in_proj'):
                    block.in_proj = LClass(block.in_proj, rank=4)
                if hasattr(block, 'out_proj'):
                    block.out_proj = LClass(block.out_proj, rank=4)
            m.load_state_dict(fc['model_state_dict'], strict=False)
            ev = evaluate(m, lbls, test_data)
            shot_evals[key] = ev
            new_acc = np.mean([ev.get(k, 0) for k in NEW_KEYWORDS])
            base_acc = np.mean([ev.get(k, 0) for k in BASE_LABELS[:10] + ['silence', 'unknown']])
            print(f"    -> test: overall={ev['_overall']:.1f}% new={new_acc:.1f}% base={base_acc:.1f}%")

In [ ]:
# ============================================================
# 12. Visualization
# ============================================================
import matplotlib.pyplot as plt

shots = [3, 5, 10, 20]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: New keyword accuracy vs shots
ax = axes[0]
for method in ['LoRA', 'NC-OPAL']:
    accs = []
    for n in shots:
        ev = shot_evals.get((method, n))
        if ev:
            accs.append(np.mean([ev.get(k, 0) for k in NEW_KEYWORDS]))
        else:
            accs.append(0)
    ax.plot(shots, accs, 'o-', label=method, linewidth=2, markersize=8)
ax.set_xlabel('Shots per keyword', fontsize=12)
ax.set_ylabel('New Keyword Accuracy (%)', fontsize=12)
ax.set_title('New Keyword Recognition', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(shots)

# Plot 2: Base class retention vs shots
ax = axes[1]
base_kws = BASE_LABELS[:10] + ['silence', 'unknown']
for method in ['LoRA', 'NC-OPAL']:
    accs = []
    for n in shots:
        ev = shot_evals.get((method, n))
        if ev:
            accs.append(np.mean([ev.get(k, 0) for k in base_kws]))
        else:
            accs.append(0)
    ax.plot(shots, accs, 's-', label=method, linewidth=2, markersize=8)
ax.set_xlabel('Shots per keyword', fontsize=12)
ax.set_ylabel('Base Class Accuracy (%)', fontsize=12)
ax.set_title('Base Class Retention (Anti-Forgetting)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(shots)

# Plot 3: Overall accuracy vs shots
ax = axes[2]
for method in ['LoRA', 'NC-OPAL']:
    accs = [shot_evals.get((method, n), {}).get('_overall', 0) for n in shots]
    ax.plot(shots, accs, '^-', label=method, linewidth=2, markersize=8)
ax.set_xlabel('Shots per keyword', fontsize=12)
ax.set_ylabel('Overall Accuracy (%)', fontsize=12)
ax.set_title('Overall (All 15 Classes)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(shots)

plt.suptitle('NC-TCN-20K Few-Shot KWS Fine-Tuning: LoRA vs NC-OPAL', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('few_shot_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSaved: few_shot_comparison.png")